# 第2章 2N7000 の伝達特性（ノートブック版）

教科書の第2章の 2-6 の解析プログラム（`fet_analysis.py`・`fet_hier.py`）を、Pico で測る代わりに**合成の伝達特性データ**で、`run_ch2.sh` の流れのとおりに1冊にまとめたものです。上から順に実行してください（Mac の Jupyter でも Google Colab でも動きます。執筆時の Mac で全部で約2分。Colab ではもっとかかることがあります）。

| 節 | 試すこと | 教科書の手順・節 | 使うファイル |
|---|---|---|---|
| 1 | 説明用のモデルで、局所的なサブスレッショルド係数 S と、S を求める範囲 | 2-3-4（図2-3） | （このノートブックの中の計算） |
| 2 | 合成の伝達特性データを作る（第1章の `iv_capture.py` と同じ形の CSV・JSON） | C の 6・8 の代わり | `synth_fet.py`（**ノートブックで足したもの**） |
| 3 | 3個の Vth（3つの方法）と S、方法による違い、行きと帰りの差 | C の 6〜9 | `fet_analysis.py` |
| 4 | S を求める範囲の上限を 3µA に下げて確かめる | C の 7 | `fet_analysis.py --sub-hi 3e-6` |
| 5 | 3個の個体差：階層ベイズ（中心化）、9回を独立とみなす誤り、τ の事前分布の感度（**ノートブックでは小さいサンプリングの設定**） | D の 10 | `fet_hier.py` |
| 6 | 非中心化の書き方と比べる | D の 10・2-3-5 | `fet_hier.py --param noncentered` |

- 次のセルたちが、2-6 の2つのファイルと、合成データを作る `synth_fet.py` を、このノートブックの隣に `%%writefile` で書き出します（教科書の `setup_ch2.sh` で `~/home-lab/ch2/` に保存する代わり）。2-6 のファイルの中身は教科書と同じです。変えたのは、図の日本語フォントの候補を足したこと（`plt.rcParams["font.family"]` の行）だけです。
- 各節では、`run_ch2.sh` の `uv run python ○○.py …` の代わりに、`run("○○.py …")` で同じ引数のまま実行します（中では `python ○○.py …` を別のプロセスで動かします）。表示を見てから、`ok()` で教科書の手順の「確かめること」と 2-7 の目安を確かめます。`ok()` は、サンプリングの設定や乱数によらずに成り立つことだけにしてあります。
- **5・6 節の PyMC のサンプリングは小さい設定**（`--draws 300 --tune 500`）を既定にしています。教科書の本番（既定の `--draws 1000 --tune 2000`、中心化で1分前後）にするときは、セルの `DRAWS`・`TUNE` を書きかえてください。小さい設定では R-hat が 1.01 を少し超えたり、発散が数回出たりすることがあり、そのとき PyMC が**英語の警告を赤い字で**表示します（`There were … divergences` や `The rhat statistic is larger than 1.01` など）。試しの設定なので出るもので、エラーではありません（ノートブックは止まりません。5 節の説明も見てください）。
- **入れていないもの（Colab ではできない手順）**：手順 A の 1〜4（印を付ける、テスタでピンの並びと抵抗・電圧を測る、Pico に第1章の `iv_tracer.py` を書き込む）、B の 5・6（配線と miniterm での動作確認）、C の 6・8 の1行目の `iv_capture.py`（Pico での約30分の測定）、D の 11（温度を変えた測り直し）。Pico 側は第1章の `iv_tracer.py`（MicroPython）・Mac 側は第1章の `iv_capture.py` をそのまま使うので、このノートブックにはコードを載せていません（第1章の 1-6 を読んでください）。測る代わりに、2 節で `iv_capture.py` と同じ形の合成データを作ります。
- ⚡ 実物の測定の注意（静電気、S と D の取り違え、ボディダイオード）は、教科書の 2-4・2-5 を読んでください。このノートブックは計算だけです。
- 数値は乱数の種を変えなければ教科書の図2-6 と同じですが、scipy・statsmodels・PyMC の版で最後の桁が変わることがあります。`ok()` はその分の幅を持たせてあります。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には numpy・scipy・pandas・matplotlib・statsmodels が最初から入っている。PyMC も入っていることが多い。版は要確認）
import importlib, subprocess, sys
for mod, pkg in [("pymc", "pymc"), ("statsmodels", "statsmodels"), ("pandas", "pandas"), ("scipy", "scipy")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import pymc, statsmodels, scipy, numpy
print("OK", "PyMC", pymc.__version__, "statsmodels", statsmodels.__version__, "scipy", scipy.__version__, "numpy", numpy.__version__)
print("（教科書は PyMC 6.4.0・statsmodels 0.15.0 で確かめた）")

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

## Pico での測定（読むだけ。このノートブックでは実行しません）

教科書では、第1章の回路の DUT の位置に 2N7000 を挿し（ゲート＝節点A、ソース＝節点K、ドレイン＝3V3(OUT)）、Pico A の第1章の `iv_tracer.py` と Mac の第1章の `iv_capture.py` で測ります。`run_ch2.sh` の1個目の2行は次のとおりです（数値は書き方の例。自分の実測値に置き換える）。

```bash
cd ~/home-lab/ch2
# 1個目：DUT = Q_A、レンジ切替 = Q_C（約30分。測定中は素子に触れない）
uv run python ../ch1/iv_capture.py --dut 2N7000_A --temp 22.5 --vref 3.296 --rl 9980 --rh 99.7 --npts 241 --reps 3 --outdir data_fet
uv run python fet_analysis.py data_fet/2N7000_A_*.csv --vdd 3.29
```

1行目が `data_fet/2N7000_A_日時.csv`（列 `dut, rep, rng, dir, idx, duty, vA, vK, vS, sdA, sdK, sdS`）と同じ名前の `.json`（`dut`・`temp_C`・`vref`・`R_L`・`R_H` など）を作り、2行目がそれを解析します。このノートブックでは、1行目の代わりに 2 節の `synth_fet.py` で同じ形のファイルを作ります。

次の3つのセルで、2-6 の `fet_analysis.py`・`fet_hier.py` と、合成データを作る `synth_fet.py`（教科書にはない、ノートブックで足したもの）を書き出します。

In [ ]:
%%writefile fet_analysis.py
#!/usr/bin/env python3
"""第2章：2N7000 の伝達特性（Id–Vgs）から Vth とサブスレッショルド係数を求める

使い方: cd ~/home-lab/ch2 && uv run python fet_analysis.py data_fet/2N7000_A_*.csv --vdd 3.29
  CSV は第1章の iv_capture.py が作ったもの（同じ名前の .json も必要）。複数まとめて指定してよい。
  --vdd     : ドレインにつないだ 3V3(OUT)（ピン36）の実測値 [V]。テスタでピン36〜ピン33 を測った値。
              第1章の iv_capture.py には VDD の引数がないので、ここで渡す。Vds の計算だけに使い、
              Vgs と Id には影響しない。渡した値は fet_summary.csv の vdd の列に記録される
  --icc     : 定電流法の電流 [A]（既定 1mA。データシートの Vth の測定条件に合わせる。onsemi 品は I_D=1mA）
  --sub-lo, --sub-hi : サブスレッショルド係数を求める電流の範囲 [A]（既定 100nA〜10µA）

第1章の回路で、DUT の G＝節点A、S＝節点K、D＝3V3(OUT) としたとき
  Vgs = vA − vK、 Id = vK / R_L + vS / R_H、 Vds = VDD − vK
結果は 1回の掃引（rep）・向き（up/down）ごとに fet_summary.csv に追記する（fet_hier.py で使う）。
"""
import argparse
import json
import pathlib
import sys

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.signal import savgol_filter

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False

K_B, Q_E = 1.380649e-23, 1.602176634e-19
I_STRONG = 20e-6        # これより大きい電流（H レンジ）を、Vth の3つの方法に使う
SUMMARY = pathlib.Path("fet_summary.csv")


def load(path):
    """CSV と JSON を読み、Vgs と Id を計算する

    第1章の式 Id = vK/R_L + vS/R_H のうち、L レンジ（レンジ切替の FET が切れている）では
    vS ≈ 0 なので vS/R_H は ADC の雑音（約0.24mV ÷ 100Ω ≈ 2µA）を足すだけになる。
    サブスレッショルドの小さな電流を測るため、L レンジでは vK/R_L だけを使い、
    さらに vK のゼロ点（Vgs < 1V でほぼ電流が流れないときの vK の中央値）を引く。
    DUT のゲートには電流が流れないので、デューティが大きいと vA は GPIO の High（≈3.3V）まで上がる。
    ADC_VREF より高いと vA は ADC の最大値に張り付き、Vgs が実際より小さく計算されるので、その点は除く。
    """
    path = pathlib.Path(path)
    df = pd.read_csv(path)
    meta = json.loads(path.with_suffix(".json").read_text(encoding="utf-8"))
    sat = df["vA"] > 0.995 * meta["vref"]                   # ADC の最大値（≈ ADC_VREF）に張り付いた点
    meta["n_sat"] = int(sat.sum())
    df = df[~sat].copy()
    df["Vgs"] = df["vA"] - df["vK"]
    zero = (df[(df["rng"] == "L") & (df["vA"] < 1.0)]
            .groupby(["rep", "dir"])["vK"].median().rename("vK0"))
    df = df.join(zero, on=["rep", "dir"])
    df["vK0"] = df["vK0"].fillna(0.0)
    is_l = df["rng"] == "L"
    df["Id"] = np.where(is_l, (df["vK"] - df["vK0"]) / meta["R_L"],
                        df["vK"] / meta["R_L"] + df["vS"] / meta["R_H"])
    return df, meta


def on_grid(d, step=0.005):
    """Vgs の小さい順に並べ、5mV 間隔の格子に直す（微分を安定させるため）"""
    d = d.sort_values("Vgs")
    v = d["Vgs"].to_numpy()
    i = np.maximum.accumulate(d["Id"].to_numpy())          # 雑音で電流が逆戻りしないように
    g = np.arange(v.min(), v.max(), step)
    return g, np.interp(g, v, i)


def deriv(y, step, win=11):
    win = min(win, len(y) - (1 - len(y) % 2))
    if win < 5:
        return np.gradient(y, step)
    return savgol_filter(y, win, 2, deriv=1, delta=step)


def vth_methods(h, icc):
    """H レンジ・強い電流の部分から、3つの方法で Vth を求める"""
    out = {"Vth_cc": np.nan, "Vth_sqrt": np.nan, "Vth_gm": np.nan, "gm_at_edge": False, "sqrt_at_edge": False}
    if len(h) < 8:
        return out, None
    g, i = on_grid(h)
    if len(g) < 8:
        return out, None
    # (1) 定電流法：Id = icc となる Vgs（log Id で補間）
    if i.min() < icc < i.max():
        out["Vth_cc"] = float(np.interp(np.log(icc), np.log(np.maximum(i, 1e-12)), g))
    # (2) √Id 外挿法：√Id の傾きが最大の点の接線を √Id = 0 まで延ばす
    s = np.sqrt(i)
    ds = deriv(s, g[1] - g[0])
    k = int(np.argmax(ds))
    out["Vth_sqrt"] = float(g[k] - s[k] / ds[k])
    out["sqrt_at_edge"] = bool(k >= len(g) - max(3, len(g) // 10))  # √Id の傾きの最大も、範囲の端なら本当の最大か分からない
    # (3) gm 最大法：gm = dId/dVgs が最大の点の接線を Id = 0 まで延ばす
    gm = deriv(i, g[1] - g[0])
    k2 = int(np.argmax(gm))
    out["Vth_gm"] = float(g[k2] - i[k2] / gm[k2])
    out["gm_at_edge"] = bool(k2 >= len(g) - max(3, len(g) // 10))  # 測った範囲の端で最大 → 本当の最大か分からない
    return out, (g, i, s, ds, k, gm, k2)


def subthreshold(l, lo, hi, vt):
    """Vgs = c + S·log10(Id) を回帰し、S [mV/dec] と理想係数 n = S / (VT ln10) を求める"""
    m = l[(l["Id"] > lo) & (l["Id"] < hi)]
    if len(m) < 5:
        return dict(S_mVdec=np.nan, S_lo=np.nan, S_hi=np.nan, n=np.nan, n_pts=len(m))
    fit = sm.OLS(m["Vgs"].to_numpy(), sm.add_constant(np.log10(m["Id"].to_numpy()))).fit()
    S = fit.params[1] * 1e3
    lo_, hi_ = fit.conf_int()[1] * 1e3
    return dict(S_mVdec=S, S_lo=lo_, S_hi=hi_, n=S / 1e3 / (vt * np.log(10)), n_pts=len(m))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("csv", nargs="+")
    ap.add_argument("--vdd", type=float, required=True, help="3V3(OUT) の実測値 [V]")
    ap.add_argument("--icc", type=float, default=1e-3)
    ap.add_argument("--sub-lo", type=float, default=1e-7)
    ap.add_argument("--sub-hi", type=float, default=1e-5)
    a = ap.parse_args()

    done = set(pd.read_csv(SUMMARY)["file"]) if SUMMARY.exists() else set()
    new_rows = []
    for path in a.csv:
        name = pathlib.Path(path).name
        df, meta = load(path)
        vt = K_B * (meta["temp_C"] + 273.15) / Q_E
        print(f"\n===== {name}（{meta['dut']}、室温 {meta['temp_C']} °C、V_T = {vt*1e3:.2f} mV）=====")
        if meta["n_sat"]:
            print(f"vA が ADC の最大値（ADC_VREF {meta['vref']} V 付近）に張り付いた {meta['n_sat']} 点を除きました")
        print(f"Vds の範囲：{(a.vdd - df['vK']).min():.2f}〜{(a.vdd - df['vK']).max():.2f} V（飽和領域にいるかの目安：Vds > Vgs − Vth）")
        fig, ax = plt.subplots(2, 2, figsize=(11, 8))
        for (rep, dirn), g in df.groupby(["rep", "dir"]):
            l = g[g["rng"] == "L"]
            h = g[(g["rng"] == "H") & (g["Id"] > I_STRONG)]
            st = subthreshold(l, a.sub_lo, a.sub_hi, vt)
            vm, curves = vth_methods(h, a.icc)
            row = dict(file=name, dut=meta["dut"], rep=rep, dir=dirn, temp_C=meta["temp_C"], vdd=a.vdd, **st, **vm)
            new_rows.append(row)
            print(f"rep {rep} {dirn:4s}: Vth 定電流 {vm['Vth_cc']:.3f} V、"
                  f"√Id外挿 {vm['Vth_sqrt']:.3f} V{'（傾きの最大が範囲の端）' if vm['sqrt_at_edge'] else ''}、"
                  f"gm最大 {vm['Vth_gm']:.3f} V{'（gm の最大が範囲の端）' if vm['gm_at_edge'] else ''}、"
                  f"S = {st['S_mVdec']:.1f} mV/dec（{st['S_lo']:.1f}〜{st['S_hi']:.1f}）、n = {st['n']:.2f}")
            if rep == 0 and dirn == "up":
                for rr, gg in g.groupby("rng"):
                    gg = gg[gg["Id"] > 0]
                    ax[0, 0].semilogy(gg["Vgs"], gg["Id"], ".", ms=3, label=f"{rr} レンジ")
                ax[0, 0].axhline(a.icc, color="gray", ls=":", lw=1)
                ax[0, 0].axhspan(a.sub_lo, a.sub_hi, color="tab:green", alpha=0.08)
                if curves:
                    gr, i, s, ds, k, gm, k2 = curves
                    ax[0, 1].plot(gr, s * 1e3, "-", label="√Id")
                    xx = np.linspace(vm["Vth_sqrt"], gr[-1], 20)
                    ax[0, 1].plot(xx, (s[k] + ds[k] * (xx - gr[k])) * 1e3, "--", label=f"接線 → {vm['Vth_sqrt']:.3f} V")
                    ax[1, 0].plot(gr, i * 1e3, "-", label="Id")
                    xx = np.linspace(vm["Vth_gm"], gr[-1], 20)
                    ax[1, 0].plot(xx, (i[k2] + gm[k2] * (xx - gr[k2])) * 1e3, "--", label=f"gm最大の接線 → {vm['Vth_gm']:.3f} V")
                    ax2 = ax[1, 0].twinx()
                    ax2.plot(gr, gm * 1e3, ":", color="tab:red")
                    ax2.set_ylabel("gm [mS]（点線）")
                lg = l[(l["Id"] > 3e-8)].sort_values("Vgs").drop_duplicates("Id")   # 同じ Id が続くと微分が無限大になる
                if len(lg) > 8:
                    ss = np.gradient(lg["Vgs"].to_numpy(), np.log10(lg["Id"].to_numpy()))
                    ok = np.isfinite(ss)
                    ss, ids = ss[ok], lg["Id"].to_numpy()[ok]
                    ax[1, 1].semilogx(ids, savgol_filter(ss, 7, 2) * 1e3 if len(ss) > 7 else ss * 1e3, ".-", ms=3)
                ax[1, 1].axhline(vt * np.log(10) * 1e3, color="gray", ls="--", lw=1)
                ax[1, 1].set_ylim(0, 300)
        ax[0, 0].set(xlabel="Vgs [V]", ylabel="Id [A]", title="伝達特性（緑の帯：S を求める範囲）")
        ax[0, 1].set(xlabel="Vgs [V]", ylabel="√Id [×10⁻³ √A]", title="√Id 外挿法")
        ax[1, 0].set(xlabel="Vgs [V]", ylabel="Id [mA]", title="gm 最大法")
        ax[1, 1].set(xlabel="Id [A]", ylabel="局所的な S [mV/dec]", title="局所的なサブスレッショルド係数（点線：n=1 の理論値）")
        for x in ax.flat:
            x.grid(True, which="both", alpha=0.3)
            if x.get_legend_handles_labels()[0]:
                x.legend(fontsize=8)
        fig.suptitle(f"{meta['dut']}（rep 0・up）")
        fig.tight_layout()
        png = pathlib.Path(path).with_suffix(".png")
        fig.savefig(png, dpi=130)
        plt.close(fig)
        print(f"{png} を保存しました")

    res = pd.DataFrame(new_rows)
    add = res[~res["file"].isin(done)]
    if len(add):
        add.to_csv(SUMMARY, mode="a", header=not SUMMARY.exists(), index=False)
        print(f"\n{SUMMARY} に {len(add)} 行を追記しました（すでにあるファイルの行は追記しない）")
    else:
        print(f"\n{SUMMARY} にはすでに同じファイルの結果があるので、追記しませんでした")

    print("\n[個体ごとのまとめ]（平均 ± 標準偏差。rep と向きをまとめたもの）")
    cols = ["Vth_cc", "Vth_sqrt", "Vth_gm", "S_mVdec", "n"]
    print(res.groupby("dut")[cols].agg(["mean", "std"]).round(3).to_string())
    print("\n[方法による違い]（同じ掃引どうしの差の平均。0 でなければ、方法によって Vth の定義が違う）")
    print(f"  √Id外挿 − 定電流：{(res['Vth_sqrt'] - res['Vth_cc']).mean()*1e3:+.0f} mV、"
          f"gm最大 − 定電流：{(res['Vth_gm'] - res['Vth_cc']).mean()*1e3:+.0f} mV")
    p = res.pivot_table(index=["dut", "rep"], columns="dir", values="Vth_cc")
    if {"up", "down"} <= set(p.columns):
        dd = (p["down"] - p["up"]).dropna() * 1e3
        print(f"\n[行きと帰りの差]（定電流法、帰り − 行き）：平均 {dd.mean():+.1f} mV、最小 {dd.min():+.1f}、最大 {dd.max():+.1f} mV"
              "（ゲート酸化膜や界面にとらえられた電荷があると差が出る。温度変化や待ち時間の不足でも出る）")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile fet_hier.py
#!/usr/bin/env python3
"""第2章：3個の 2N7000 の個体差を、ベイズ階層モデルで推定する

使い方: cd ~/home-lab/ch2 && uv run python fet_hier.py fet_summary.csv
  fet_summary.csv は fet_analysis.py が作ったもの（掃引ごと・向きごとの Vth と S）
  --param noncentered : 非中心化した書き方で当てはめる（既定は centered＝中心化。2-3-5）
  --draws, --tune     : サンプリングの点数（既定 1000・2000。試しに速く動かすときは 300・500 など）

モデル（パラメータごとに別々に当てはめる。y は「1回の掃引で求めた値」、行きと帰りは平均する）
  y_jk ~ Normal(θ_j, σ)          … 個体 j の k 回目の測定。σ は測り直しのばらつき
  θ_j  ~ Normal(μ, τ)            … 個体ごとの真の値（中心化した書き方。既定）
         （非中心化：θ_j = μ + τ·z_j、z_j ~ Normal(0, 1)。--param noncentered）
  μ    ~ Normal(事前の中心, 事前の幅)   … 「2N7000 という型番」の平均
  τ    ~ HalfNormal(尺度)            … 個体どうしのばらつき（3個ではほとんど決まらない）
  σ    ~ HalfNormal(尺度)
τ の事前分布を2通りにして、結果がどれだけ事前分布に左右されるか（感度）を比べる。
"""
import argparse
import warnings

import numpy as np
import pandas as pd
import pymc as pm
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)

# パラメータごとの事前分布の設定（中心, 幅, σ の尺度, τ の尺度の2通り）
# Vth の中心と幅は、2N7000 のデータシートの範囲（onsemi 品で 0.8〜3.0V、標準 2.1V。V_DS=V_GS、I_D=1mA）を覆うように広くとる
SETTINGS = {
    "Vth_cc": dict(unit="V", mu=(2.0, 1.0), sigma=0.05, taus=(0.5, 0.05)),
    "S_mVdec": dict(unit="mV/dec", mu=(90.0, 40.0), sigma=10.0, taus=(30.0, 3.0)),
}


def fit(y, idx, J, mu_prior, sig_scale, tau_scale, seed, param, draws, tune):
    with pm.Model():
        mu = pm.Normal("mu", mu_prior[0], mu_prior[1])
        tau = pm.HalfNormal("tau", tau_scale)
        sigma = pm.HalfNormal("sigma", sig_scale)
        if param == "centered":
            # 中心化：各個体が測り直しでよく決まる（σ/√3 が τ よりずっと小さい）ときに効率がよい
            theta = pm.Normal("theta", mu, tau, shape=J)
        else:
            # 非中心化：1個あたりのデータが少なく、θ が τ の事前分布で決まるようなときに安定する
            z = pm.Normal("z", 0, 1, shape=J)
            theta = pm.Deterministic("theta", mu + tau * z)
        pm.Normal("y", theta[idx], sigma, observed=y)
        pm.Deterministic("theta_new", mu + tau * pm.Normal("z_new", 0, 1))   # 4個目の予測
        return pm.sample(draws, tune=tune, chains=4, cores=1, target_accept=0.99,
                         random_seed=seed, progressbar=False)


def rhat(x):
    """分割した R-hat（Gelman–Rubin）。x は (チェーン数, 1チェーンの点数) の配列。1.01 未満が目安"""
    c, n = x.shape
    h = n // 2
    x = np.concatenate([x[:, :h], x[:, h:2 * h]])         # 各チェーンを前半・後半に分ける
    m, n = x.shape
    w = x.var(axis=1, ddof=1).mean()
    b = n * x.mean(axis=1).var(ddof=1)
    return float(np.sqrt(((n - 1) / n * w + b / n) / w))


def summarize(idata, name):
    """事後分布の平均と、95% 信用区間（2.5%点〜97.5%点）、R-hat"""
    x = idata.posterior[name].values                      # (チェーン, 点, …)
    out = []
    for k in ([None] if x.ndim == 2 else range(x.shape[2])):
        xx = x if k is None else x[:, :, k]
        out.append((float(xx.mean()), *np.percentile(xx, [2.5, 97.5]), rhat(xx)))
    return out


def main(path, param, draws, tune):
    d = pd.read_csv(path)
    print(f"書き方：{'中心化' if param == 'centered' else '非中心化'}（--param {param}）")
    for col, cfg in SETTINGS.items():
        g = d.groupby(["dut", "rep"], as_index=False)[col].mean().dropna()   # 行きと帰りを平均
        duts = sorted(g["dut"].unique())
        J = len(duts)
        idx = g["dut"].map({k: i for i, k in enumerate(duts)}).to_numpy()
        y = g[col].to_numpy()
        u = cfg["unit"]
        print(f"\n==================== {col}（{J} 個、測定 {len(y)} 回）====================")
        for k in duts:
            v = g.loc[g["dut"] == k, col]
            print(f"  {k}: " + ", ".join(f"{x:.4g}" for x in v))
        # まず、よくある「まちがい」：9回を独立な9個とみなした平均の区間
        n = len(y)
        h = stats.t.ppf(0.975, n - 1) * y.std(ddof=1) / np.sqrt(n)
        m = g.groupby("dut")[col].mean().to_numpy()
        h2 = stats.t.ppf(0.975, J - 1) * m.std(ddof=1) / np.sqrt(J) if J >= 2 else np.nan
        print(f"\n[誤り] 全 {n} 回を独立とみなした平均の95%区間：{y.mean():.4g} ± {h:.3g} {u}")
        print(f"[正しい] 個体の平均 {J} 個から求めた95%区間：{m.mean():.4g} ± {h2:.3g} {u}（自由度 {J-1}、t = {stats.t.ppf(0.975, J-1):.2f}）")
        for tau_scale, seed in zip(cfg["taus"], (1, 2)):
            idata = fit(y, idx, J, cfg["mu"], cfg["sigma"], tau_scale, seed, param, draws, tune)
            div = int(idata.sample_stats["diverging"].values.sum())
            rows = [("μ（型番の平均）", summarize(idata, "mu")[0]),
                    ("τ（個体のばらつき）", summarize(idata, "tau")[0]),
                    ("σ（測り直しのばらつき）", summarize(idata, "sigma")[0])]
            rows += [(f"θ（{k}）", r) for k, r in zip(duts, summarize(idata, "theta"))]
            rows += [("4個目の予測", summarize(idata, "theta_new")[0])]
            print(f"\n--- τ の事前分布 HalfNormal({tau_scale}) ---（発散 {div} 回、"
                  f"R-hat の最大 {max(r[3] for _, r in rows):.3f}）")
            for lab, (mean, lo, hi, _) in rows:
                print(f"  {lab:22s} 平均 {mean:9.4g}、95%信用区間 {lo:9.4g} 〜 {hi:9.4g} {u}")
            if div > 0:
                other = "noncentered" if param == "centered" else "centered"
                print("  （発散が出た：少数なら結果の傾向は使えるが、"
                      f"--param {other} で書き方を替えて比べるか、τ の事前分布を見直す）")
        print("\n・θ（各個体）と σ はデータでよく決まり、事前分布を変えてもほとんど変わらない。")
        print("・τ と「4個目の予測」は、τ の事前分布を変えると大きく変わる → 3個のデータでは決まらない。")


if __name__ == "__main__":
    ap = argparse.ArgumentParser(description="3個の 2N7000 の個体差を階層ベイズで推定する")
    ap.add_argument("summary", help="fet_analysis.py が作った fet_summary.csv")
    ap.add_argument("--param", choices=["centered", "noncentered"], default="centered",
                    help="階層モデルの書き方（既定 centered＝中心化）")
    ap.add_argument("--draws", type=int, default=1000)
    ap.add_argument("--tune", type=int, default=2000)
    a = ap.parse_args()
    main(a.summary, a.param, a.draws, a.tune)

In [ ]:
%%writefile synth_fet.py
#!/usr/bin/env python3
"""ノートブック版で足したもの：第1章の iv_capture.py と同じ形の CSV・JSON を、EKV 型のモデルから作る

使い方: python synth_fet.py        （data_fet/ に 2N7000_A・B・C の3個分を作る）
  モデル（2-8 の発展課題2 の式に、強反転での移動度の低下 1/(1 + θ(Vgs − Vth)) を掛けたもの）
    Id = 2nβV_T²·ln²(1 + exp((Vgs − Vth)/(2nV_T))) / (1 + θ·max(Vgs − Vth, 0)) + 0.2nA（漏れ）
  第1章の回路：ゲート＝節点A、ソース＝節点K。L レンジは R_L だけ、H レンジは R_L と (R_ON + R_H) の並列を
  ソースと GND の間に入れたものとして、vK = Id(vA − vK)·R を二分法で解く（ソースの抵抗による負帰還）
  個体の真の値：A Vth 2.05V・n 1.45、B 2.17V・1.52、C 1.97V・1.40（25℃、V_T = 25.69mV）
  掃引（rep）ごとに Vth が 4mV ほどゆらぎ、帰り（down）は 6mV 右にずれる（ヒステリシスのつもり）
  ADC の雑音は 0.24mV（256回平均のつもり）。デューティが大きいと vA は ADC_VREF を超え、fet_analysis.py が除く
  点の数は 121（教科書の run_ch2.sh の --npts 241 より少なくして、速く作る）
"""
import csv
import json
import pathlib

import numpy as np

rng = np.random.default_rng(5)
out = pathlib.Path("data_fet")
out.mkdir(exist_ok=True)
VT = 0.02569                                   # 25℃ の熱電圧 [V]
RL, RH, RON = 9980.0, 99.7, 6.0                # R_L・R_H と、レンジ切替の FET のオン抵抗 [Ω]
VREF = 3.296


def idrain(vgs, vth, n, beta=0.2, theta=0.4):
    x = (vgs - vth) / (2 * n * VT)
    f = np.log1p(np.exp(np.clip(x, -60, 60))) ** 2
    return 2 * n * beta * VT ** 2 * f / (1 + theta * np.maximum(vgs - vth, 0)) + 2e-10


def solve(vA, reff, vth, n):
    """vK = Id(vA − vK)·reff を二分法で解く"""
    lo, hi = 0.0, vA
    for _ in range(80):
        vK = (lo + hi) / 2
        if idrain(vA - vK, vth, n) * reff > vK:
            lo = vK
        else:
            hi = vK
    return vK


devs = {"2N7000_A": (2.05, 1.45), "2N7000_B": (2.17, 1.52), "2N7000_C": (1.97, 1.40)}
for name, (vth0, n0) in devs.items():
    rows = []
    npts = 121
    for rep in range(3):
        vth = vth0 + rng.normal(0, 0.004)                     # 掃引ごとのゆらぎ
        for r in ("L", "H"):
            reff = RL if r == "L" else 1 / (1 / RL + 1 / (RON + RH))
            up = [round(65535 * i / (npts - 1)) for i in range(npts)]
            for idx, d in enumerate(up + up[::-1][1:]):
                vt = vth + (0.006 if idx >= npts else 0)          # 帰りで少し右にずれる（ヒステリシス）
                vA = d / 65535 * 3.30 + rng.normal(0, 2e-4)
                vK = solve(vA, reff, vt, n0)
                vS = 0.0 if r == "L" else vK * RH / (RON + RH)
                f = lambda v: max(0.0, v + rng.normal(0, 2.4e-4))
                rows.append(dict(dut=name, rep=rep, rng=r, dir="up" if idx < npts else "down", idx=idx, duty=d,
                                 vA=f(vA), vK=f(vK), vS=f(vS), sdA=0.0019, sdK=0.0019, sdS=0.0019))
    base = out / f"{name}_20261201-1000{list(devs).index(name)}"
    with open(base.with_suffix(".csv"), "w", newline="") as fp:
        w = csv.DictWriter(fp, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)
    base.with_suffix(".json").write_text(json.dumps(dict(
        dut=name, wavelength_nm=None, temp_C=25.0, vref=VREF, R_L=RL, R_H=RH, R_F=1000.0,
        npts=npts, settle_ms=600, nsamp=64, reps=3, datetime="（合成データ）")))
    print(f"{base}.csv と .json を作りました（{len(rows)} 行）")

In [ ]:
import glob, os, re, shlex, shutil, subprocess, sys
from IPython.display import Image, display


def run(args, cwd=None):
    """ターミナルの「uv run python ○○.py …」の代わり。表示を出し、あとで確かめられるように文字列で返す"""
    here = cwd or "."
    words = []
    for w in shlex.split(args):   # data_fet/2N7000_A_*.csv のような * を、ターミナルと同じく名前の順に広げる
        hits = sorted(glob.glob(os.path.join(here, w))) if any(c in w for c in "*?[") else []
        words += [os.path.relpath(x, here) for x in hits] or [w]
    script = os.path.abspath(words[0]) if os.path.exists(words[0]) else words[0]
    p = subprocess.run([sys.executable, script] + words[1:], capture_output=True, text=True, cwd=cwd)
    print(p.stdout, end="")
    err = "\n".join(l for l in p.stderr.splitlines()
                    if "findfont" not in l and not re.match(r"(Initializing NUTS|Sequential sampling|NUTS:|Sampling \d)", l))
    if err.strip():
        print(err, file=sys.stderr)   # PyMC の「R-hat が大きい」「発散」などの注意はそのまま出す
    assert p.returncode == 0, "★実行に失敗：" + args
    return p.stdout


def show(*paths, width=760):
    """保存された図を表示する"""
    for f in paths:
        print(f)
        display(Image(filename=f, width=width))


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def near(got, want, d):
    """got が want と ±d 以内か（d は表示の最後の桁の 1 つ分。numpy・scipy の版で最後の桁が変わる分の幅）。リストなら要素ごと（d もリストにできる）"""
    if isinstance(want, (list, tuple)):
        got = list(got)
        ds = d if isinstance(d, (list, tuple)) else [d] * len(want)
        return len(got) == len(want) and all(near(g, w, e) for g, w, e in zip(got, want, ds))
    return abs(float(got) - want) <= d * (1 + 1e-6)


def num(pattern, text, i=1):
    """表示（text）から、pattern の (…) の数を取り出す"""
    m = re.search(pattern, text)
    assert m, "表示に見つからない：" + pattern
    return float(m.group(i))


def hier(text):
    """fet_hier.py の表示を {(パラメータ, τ の尺度): {行の名前: (平均, 下, 上)}} にする"""
    out = {}
    for blk in re.split(r"^=+ ", text, flags=re.M)[1:]:
        col = blk.split("（")[0]
        for sub in re.split(r"^--- τ の事前分布 HalfNormal\(", blk, flags=re.M)[1:]:
            scale = float(sub.split(")")[0])
            rows = re.findall(r"^  (\S+)\s+平均\s+([-\d.e+]+)、95%信用区間\s+([-\d.e+]+) 〜\s+([-\d.e+]+)", sub, re.M)
            out[(col, scale)] = {k: tuple(map(float, v)) for k, *v in rows}
            out[(col, scale)]["発散"] = int(re.search(r"発散 (\d+) 回", sub).group(1))
            out[(col, scale)]["R-hat"] = float(re.search(r"R-hat の最大 ([\d.]+)", sub).group(1))
    return out


# やり直したとき：fet_summary.csv は追記になるので、先に消す（教科書の 2-7「古い結果を消したい」と同じ）
for f in ["fet_summary.csv"]:
    if os.path.exists(f):
        os.remove(f)
shutil.rmtree("subhi", ignore_errors=True)
os.environ.setdefault("PYTENSOR_FLAGS", "compiledir=" + os.path.abspath("pytensor_cache"))   # PyMC のコンパイルの置き場所（このフォルダの中）
print("準備ができました")

## 1. 説明用のモデルで、局所的な S と S を求める範囲（2-3-4、図2-3）

図2-3 と同じ説明用のモデル（2-8 の EKV 型の式に、強反転での移動度の低下を掛けたもの。$V_{th} = 2.0$V、$n = 1.5$、$\beta = 0.2$、$\theta = 0.3$、$V_T = 25.7$mV（25℃））で、局所的なサブスレッショルド係数 $S = dV_{GS}/d\log_{10} I_D$ を電流ごとに計算し、$V_{GS} = c + S\log_{10} I_D$ を範囲を変えて回帰します（この節は Pico のデータを使わない、ノートブックの中だけの計算です）。

弱反転では $S = nV_T\ln 10 \approx 88.8$mV/dec で平らになり、電流が増えると大きくなります。100nA〜10µA で回帰すると約3% 大きく出て、上限を 3µA に下げると理論値に近づきます。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False
import logging
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)   # 候補のうち見つからないフォントの警告は出さない（日本語のフォントがあるかは、0-1 のセルの表示で分かる）

VT, N, VTH, BETA, THETA = 0.0257, 1.5, 2.0, 0.2, 0.3


def ekv(v):
    x = (v - VTH) / (2 * N * VT)
    return 2 * N * BETA * VT ** 2 * np.log1p(np.exp(np.minimum(x, 60))) ** 2 / (1 + THETA * np.maximum(v - VTH, 0))


vg = np.arange(1.0, 3.0, 0.0005)
idr = ekv(vg)
S_local = np.gradient(vg, np.log10(idr)) * 1e3            # 局所的な S [mV/dec]


def fitS(lo, hi):
    m = (idr >= lo) & (idr <= hi)
    return np.polyfit(np.log10(idr[m]), vg[m], 1)[0] * 1e3


S_ideal = N * VT * np.log(10) * 1e3
S_def, S_3u = fitS(1e-7, 1e-5), fitS(1e-7, 3e-6)
print(f"nV_T ln10 = {S_ideal:.1f} mV/dec、100nA〜10µA で回帰 {S_def:.1f} mV/dec、100nA〜3µA で回帰 {S_3u:.1f} mV/dec")
for target in [1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3]:
    k = np.argmin(abs(np.log(idr / target)))
    print(f"  Id = {target:.0e} A：Vgs = {vg[k]:.3f} V、局所的な S = {S_local[k]:.1f} mV/dec")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].semilogy(vg, idr)
ax[0].axhspan(1e-7, 1e-5, color="tab:green", alpha=0.15)
ax[0].set(xlabel="Vgs [V]", ylabel="Id [A]", ylim=(1e-10, 1e-1), title="(a) 伝達特性（緑の帯：S を求める既定の範囲）")
ax[1].semilogx(idr, S_local)
ax[1].axhline(S_ideal, color="gray", ls="--")
ax[1].axvspan(1e-7, 1e-5, color="tab:green", alpha=0.15)
ax[1].set(xlabel="Id [A]", ylabel="局所的な S [mV/dec]", xlim=(1e-10, 1e-2), ylim=(0, 300), title="(b) 局所的な S（破線：nV_T ln10）")
for x in ax:
    x.grid(True, which="both", alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# 確かめ（2-3-4 と図2-3）
ok(abs(S_ideal - 88.8) < 0.05, f"弱反転の S は nV_T ln10 ≈ 88.8 mV/dec（{S_ideal:.2f}）")
k = np.argmin(abs(np.log(idr / 1e-9)))
ok(abs(S_local[k] - S_ideal) < 0.5, "1nA では局所的な S が理論値で平ら")
ok(near(S_def, 91.3, 0.1) and near(S_3u, 90.6, 0.1), "100nA〜10µA で回帰すると 91.3 mV/dec、上限を 3µA に下げると 90.6 mV/dec（図2-3）")
ok(0.02 < S_def / S_ideal - 1 < 0.04, f"既定の範囲では約3% 大きく出る（{(S_def / S_ideal - 1) * 100:.1f}%）")
ok(np.all(np.diff(S_local[(idr > 1e-7) & (idr < 1e-3)]) > 0), "強反転に近づくと局所的な S は大きくなっていく")

## 2. 合成の伝達特性データを作る（手順 C の 6・8 の `iv_capture.py` の代わり）

`synth_fet.py` は、3個の 2N7000（真の値 A：$V_{th}$ 2.05V・$n$ 1.45、B：2.17V・1.52、C：1.97V・1.40）を第1章の回路で3回ずつ（行き・帰り、L・H レンジ）測ったつもりのデータを、`iv_capture.py` と同じ列の CSV と JSON で `data_fet/` に作ります。ソースの抵抗による負帰還（L レンジは $R_L$、H レンジは $R_L$ と $R_{ON}+R_H$ の並列）、ADC の雑音、掃引ごとの数mV のゆらぎ、帰りの 6mV のずれ（ヒステリシスのつもり）を入れてあります。これは教科書の図2-6 を描いたのと同じ合成データです。

In [ ]:
run("synth_fet.py")

In [ ]:
# 確かめ：iv_capture.py と同じ形のファイルができた
import json
import pandas as pd
csvs = sorted(glob.glob("data_fet/2N7000_*.csv"))
ok(len(csvs) == 3 and all(os.path.exists(c[:-4] + ".json") for c in csvs), "3個分の CSV と同じ名前の JSON ができた")
d0 = pd.read_csv(csvs[0])
ok(list(d0.columns) == "dut rep rng dir idx duty vA vK vS sdA sdK sdS".split(), "CSV の列は第1章の iv_capture.py と同じ")
ok({"dut", "temp_C", "vref", "R_L", "R_H"} <= set(json.load(open(csvs[0][:-4] + ".json"))), "JSON に dut・temp_C・vref・R_L・R_H がある")

## 3. 3個を解析する（手順 C の 6〜9）

`run_ch2.sh` の各個体の2行目（`fet_analysis.py data_fet/2N7000_○_*.csv --vdd 3.29`）を、A・B・C の順に実行します。各個体で 6行（3回×行き・帰り）の Vth（定電流法・√Id 外挿法・gm 最大法）と S が表示され、`fet_summary.csv` に追記され、図が `data_fet/2N7000_○_….png` に保存されます。「vA が ADC の最大値に張り付いた点を除きました」「gm の最大が範囲の端」と表示されるのは異常ではありません（2-3-2・2-3-3）。

In [ ]:
out_A = run("fet_analysis.py data_fet/2N7000_A_*.csv --vdd 3.29")
show(glob.glob("data_fet/2N7000_A_*.png")[0])

In [ ]:
out_B = run("fet_analysis.py data_fet/2N7000_B_*.csv --vdd 3.29")

In [ ]:
out_C = run("fet_analysis.py data_fet/2N7000_C_*.csv --vdd 3.29")

In [ ]:
# 確かめ（手順 C の 7〜9 と 2-7）
summ = pd.read_csv("fet_summary.csv")
ok(len(summ) == 18 and summ.groupby("dut").size().tolist() == [6, 6, 6], "fet_summary.csv に 18行（3個×3回×2方向）がそろった")
spread = summ.groupby("dut")["Vth_cc"].agg(lambda x: x.max() - x.min()) * 1e3
ok((spread < 20).all(), "各個体の6回の Vth（定電流法）の差が数mV〜十数mV 以内 " + str(spread.round(1).to_dict()))
ok((summ["S_mVdec"] > 59.2).all() and (summ["n"] > 1).all(), f"S がどれも 59mV/dec より大きい（n > 1）：{summ.S_mVdec.min():.1f}〜{summ.S_mVdec.max():.1f} mV/dec")
ok(summ["S_mVdec"].between(80, 120).all(), "S は 80〜120 mV/dec の範囲（2-7 の目安）")
d_sqrt = num(r"√Id外挿 − 定電流：([-+\d]+) mV", out_C)
d_gm = num(r"gm最大 − 定電流：([-+\d]+) mV", out_C)
ok(d_sqrt < 0 < d_gm and 20 <= abs(d_sqrt) <= 200 and 20 <= abs(d_gm) <= 200,
   f"√Id 外挿法は定電流法より低め（{d_sqrt:+.0f} mV）、gm 最大法は高め（{d_gm:+.0f} mV）で、差は数十〜百数十mV")
ok(all("gm の最大が範囲の端" in o for o in (out_A, out_B, out_C)), "「gm の最大が範囲の端」と表示される（強反転を十分に測れない回路なので）")
ok(all("張り付いた" in o for o in (out_A, out_B, out_C)), "「ADC の最大値に張り付いた点を除きました」と表示される（異常ではない）")
hys = num(r"帰り − 行き）：平均 ([-+\d.]+) mV", out_C)
ok(3 <= hys <= 9, f"行きと帰りの差は数mV（{hys:+.1f} mV。合成データでは帰りを 6mV ずらした）")
m = summ.groupby("dut")["Vth_cc"].mean()
ok(near(m.tolist(), [2.159, 2.285, 2.075], 0.001), "個体ごとの Vth（定電流法）の平均 A 2.159・B 2.285・C 2.075 V（図2-6）")

## 4. S を求める範囲の上限を下げて確かめる（手順 C の 7）

`--sub-hi 3e-6` にして解析し直し、S がほとんど（1〜2% 以内）変わらないことを確かめます。教科書では `fet_summary.csv` を消してから解析し直しますが、ここでは 5 節で使う結果を残しておくため、`subhi/` に写したデータで実行します（図も `subhi/data_fet/` に保存されます）。

In [ ]:
os.makedirs("subhi/data_fet", exist_ok=True)
for f in glob.glob("data_fet/*.csv") + glob.glob("data_fet/*.json"):
    shutil.copy(f, "subhi/data_fet/")
out_sh = run("fet_analysis.py data_fet/2N7000_*.csv --vdd 3.29 --sub-hi 3e-6", cwd="subhi")

In [ ]:
# 確かめ（手順 C の 7）
s0 = summ.groupby("dut")["S_mVdec"].mean()
s1 = pd.read_csv("subhi/fet_summary.csv").groupby("dut")["S_mVdec"].mean()
ch = (s1 / s0 - 1) * 100
print(pd.DataFrame({"既定（〜10µA）": s0.round(1), "--sub-hi 3e-6": s1.round(1), "変化 [%]": ch.round(2)}))
ok((ch.abs() <= 2).all(), "上限を 3µA に下げても、S の変化は 2% 以内")

## 5. 3個の個体差：階層ベイズモデル（2-3-5、手順 D の 10）

`fet_hier.py fet_summary.csv` で、Vth（定電流法）と S のそれぞれについて、

- 【誤り】9回を独立な9個とみなした平均の95%区間（自由度8、$t \approx 2.31$）と、【正しい】個体の平均3個から求めた区間（自由度2、$t \approx 4.30$）
- 階層モデル（既定の中心化）を、τ の事前分布 2通り（広い・狭い）で当てはめた結果

を表示します。**ノートブックの既定は小さいサンプリングの設定**（`--draws 300 --tune 500`、執筆時の Mac で約15秒）です。教科書の本番は `DRAWS, TUNE = 1000, 2000`（`fet_hier.py` の既定。1分前後）です。

**赤い字の英語の警告について**：このセルと 6 節のセルでは、表示の途中に `There were 3 divergences after tuning. Increase target_accept or reparameterize.`（調整の後に発散が 3 回あった）や `The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling.`（R-hat が 1.01 より大きいパラメータがある）のような英語の警告が、赤い背景で出ることがあります（文と回数は PyMC の版と乱数で変わります）。これは**試しの設定でサンプルが少ないために出るもので、エラーではありません**。セルは止まらず、下の確かめ（`ok()`）はこの警告が出ても成り立つことだけにしてあります。本番の設定（`DRAWS, TUNE = 1000, 2000`）で R-hat の最大が 1.01 前後・発散が 0 回か少数に収まるか（本文の手順 D の 10 の確認）を、自分の Mac で確かめてください。本番の設定でも警告が続くときは、本文の 2-3-5（中心化と非中心化）を見ます。

In [ ]:
DRAWS, TUNE = 300, 500   # 試しの設定。教科書の本番（手順 D の 10）は DRAWS, TUNE = 1000, 2000（fet_hier.py の既定。中心化で1分前後）
out_h = run(f"fet_hier.py fet_summary.csv --draws {DRAWS} --tune {TUNE}")

In [ ]:
# 確かめ（手順 D の 10、2-3-5、図2-6。サンプリングの設定によらず成り立つこと）
h = hier(out_h)
w9 = num(r"\[誤り\] 全 9 回を独立とみなした平均の95%区間：([\d.]+) ± ([\d.]+) V", out_h, 2)
w3 = num(r"\[正しい\] 個体の平均 3 個から求めた95%区間：([\d.]+) ± ([\d.]+) V", out_h, 2)
ok(near(num(r"\[正しい\] 個体の平均 3 個から求めた95%区間：([\d.]+)", out_h), 2.173, 0.001) and near(w9 * 1e3, 70, 1) and near(w3 * 1e3, 262, 1),
   "Vth の平均 2.173 V、9回を独立とみなすと ±70 mV、個体の平均3個からは ±262 mV（図2-6）")
ok(3 < w3 / w9 < 5, f"正しい区間は約4倍広い（{w3 / w9:.1f}倍）")
g = summ.groupby(["dut", "rep"], as_index=False)["Vth_cc"].mean()
sig = np.sqrt(g.groupby("dut")["Vth_cc"].var().mean())
ok(0.002 < sig < 0.005 and abs(g.groupby("dut")["Vth_cc"].mean().std() - 0.106) < 0.001,
   f"測り直しのばらつき σ ≈ 3mV（{sig * 1e3:.1f} mV）、個体の平均の標準偏差 約106mV（図2-6）")
for col, (wide, narrow) in [("Vth_cc", (0.5, 0.05)), ("S_mVdec", (30.0, 3.0))]:
    a, b = h[(col, wide)], h[(col, narrow)]
    th = [k for k in a if k.startswith("θ")]
    sc = 1.0 if col == "Vth_cc" else 1e-3
    dth = max(abs(a[k][0] - b[k][0]) for k in th)
    ok(len(th) == 3 and dth < (0.005 if col == "Vth_cc" else 1.0),
       f"{col}：θ（各個体）は τ の事前分布を変えてもほとんど変わらない（平均の差の最大 {dth:.3g}）")
    wt = lambda r, k: r[k][2] - r[k][1]
    ok(wt(b, "τ（個体のばらつき）") < 0.6 * wt(a, "τ（個体のばらつき）") and wt(b, "4個目の予測") < 0.6 * wt(a, "4個目の予測"),
       f"{col}：τ と「4個目の予測」の区間は、τ の事前分布を狭くすると大きく縮む"
       f"（4個目の予測の幅 {wt(a, '4個目の予測'):.3g} → {wt(b, '4個目の予測'):.3g}）")
ok(h[("Vth_cc", 0.5)]["σ（測り直しのばらつき）"][2] < 0.02, "σ はデータでよく決まる（95%区間の上が 20mV より小さい）")
print("（R-hat と発散は設定と乱数で変わるので、確かめには入れていない：",
      {f"{c} τ尺度{s}": (v["R-hat"], v["発散"]) for (c, s), v in h.items()}, "）")

## 6. 非中心化の書き方と比べる（2-3-5、2-7 の表の「発散が多い」）

2-3-5 のとおり、この章のデータのように各個体が測り直しでよく決まる（$\sigma/\sqrt3 \approx 2$mV が個体のばらつき約100mV よりずっと小さい）ときは中心化のほうが効率がよく、非中心化では R-hat が大きくなったり、発散が出たり、時間が何倍もかかったりします。同じ小さい設定で `--param noncentered` を当てはめ、θ が同じになるか、R-hat・発散・時間がどうかを比べます（執筆時の Mac で約40秒。本番の設定では10分前後かかることがあります）。

In [ ]:
import time
t0 = time.time()
out_nc = run(f"fet_hier.py fet_summary.csv --draws {DRAWS} --tune {TUNE} --param noncentered")   # 本番は DRAWS, TUNE = 1000, 2000
t_nc = time.time() - t0

In [ ]:
# 確かめ：書き方を替えても、θ（各個体）は同じになる（R-hat・発散・時間は設定と乱数で変わるので表示だけ）
hn = hier(out_nc)
for key in h:
    th = [k for k in h[key] if k.startswith("θ")]
    tol = 0.005 if key[0] == "Vth_cc" else 1.0
    ok(max(abs(h[key][k][0] - hn[key][k][0]) for k in th) < tol, f"{key[0]}・τ の尺度 {key[1]}：中心化と非中心化で θ の平均が同じ（差 {tol} 未満）")
print(pd.DataFrame({"中心化 R-hat": [v["R-hat"] for v in h.values()], "中心化 発散": [v["発散"] for v in h.values()],
                    "非中心化 R-hat": [v["R-hat"] for v in hn.values()], "非中心化 発散": [v["発散"] for v in hn.values()]},
                   index=[f"{c}・HalfNormal({s})" for c, s in h]))
print(f"非中心化の時間 {t_nc:.0f} 秒（中心化は 5 節で約15秒。Mac での目安）")

## おわりに

- 説明用のモデルで、弱反転の S が $nV_T\ln10$ で平らになり、強反転に近づくと大きくなること、S を求める範囲の上限で値が数% 変わることを確かめました。
- 合成の伝達特性データで、Vth の3つの方法が数十〜百数十mV 違うこと（定義が違う）、行きと帰りの差、S を求める範囲を変えても S がほとんど変わらないことを確かめました。
- 階層ベイズでは、9回を独立とみなすと区間が約4分の1に狭くなる誤りと、3個しかないと τ と「4個目の予測」が事前分布で大きく変わる（データからは決まらない）ことを見ました。
- 5・6 節を教科書の本番の設定（`DRAWS, TUNE = 1000, 2000`）にすると、R-hat は 1.01 前後になります。
- このノートブックに入れていないもの：Pico での伝達特性の測定（第1章の `iv_tracer.py`・`iv_capture.py`）、テスタでの確認、温度を変えた測り直し。これらは Mac と Pico で教科書の手順のとおりに行います。自分の測定の CSV・JSON を Colab の左のファイルの欄の `data_fet/` に上げれば、3 節以降は同じように動くはずです（執筆時には確かめていません）。